# Building Analytics Marts

Rebuilds all five marts from scratch using the merged fact table (`final.csv`).

| Mart | Grain | Purpose |
|---|---|---|
| `mart_revenue` | 1 row = 1 month | Monthly financial & user KPIs |
| `mart_carrier` | 1 row = month × carrier | Carrier-level performance & share |
| `mart_behavior` | 1 row = month × Type_user × Purchase_status | Mua hộ vs normal, New vs Current |
| `mart_demographic` | 1 row = 1 user | Slice by Age / Gender / Location / Cohort in PBI |
| `mart_simulation(Optional)` | 1 row = 1 carrier × scenario | Cashback policy what-if analysis |

## 1. Setup & Load

In [1]:
import pandas as pd
import os
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent
SRC_PATH   = BASE_DIR / "Data" / "processed" / "final.csv"
MART_DIR   = BASE_DIR / "Data" / "mart"
MART_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_csv(SRC_PATH)
df['Date']           = pd.to_datetime(df['Date'])
df['First_tran_date']= pd.to_datetime(df['First_tran_date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13494 entries, 0 to 13493
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   user_id          13494 non-null  int64         
 1   order_id         13494 non-null  int64         
 2   Date             13494 non-null  datetime64[ns]
 3   Amount           13494 non-null  int64         
 4   Merchant_id      13494 non-null  int64         
 5   Purchase_status  13494 non-null  int64         
 6   Merchant_name    13494 non-null  object        
 7   Rate_pct         13494 non-null  int64         
 8   Revenue          13494 non-null  float64       
 9   First_tran_date  13494 non-null  datetime64[ns]
 10  Location         13494 non-null  object        
 11  Age              13494 non-null  object        
 12  Gender           13494 non-null  object        
 13  Type_user        13494 non-null  object        
dtypes: datetime64[ns](2), float64(1), int6

## 2. Derived Columns


### Time features

In [7]:
# month as 'YYYY-MM' string (sortable)
df['month']        = df['Date'].dt.to_period('M').astype(str)

# cohort month from first transaction
df['cohort_month'] = df['First_tran_date'].dt.to_period('M').astype(str)

# cohort year from first transaction
df['cohort_year']  = df['First_tran_date'].dt.year

# Weekday (0 = Monday, 6 = Sunday)
df['weekday'] = df['Date'].dt.weekday

# Weekend flag (1 = weekend, 0 = weekday)
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)

# Day of month (1–31)
df['day_of_month'] = df['Date'].dt.day

print(df[['month','cohort_month','cohort_year','weekday','is_weekend','day_of_month']].head())

     month cohort_month  cohort_year  weekday  is_weekend  day_of_month
0  2020-01      2018-12         2018        2           0             1
1  2020-01      2019-12         2019        2           0             1
2  2020-01      2019-11         2019        2           0             1
3  2020-01      2019-10         2019        2           0             1
4  2020-01      2019-12         2019        2           0             1


### business features

In [8]:
# carrier alias (Merchant_name is the carrier)
df['carrier']      = df['Merchant_name']

# human-readable purchase label
df['purchase_label'] = df['Purchase_status'].map({1: 'Mua hộ', 0: 'Khác'})

# carrier_group: merge Gmobile into Others for carrier mart
df['carrier_group'] = df['carrier'].replace({'Gmobile': 'Others'})

print(df[['carrier','purchase_label','carrier_group']].head())

    carrier purchase_label carrier_group
0  Mobifone           Khác      Mobifone
1  Mobifone           Khác      Mobifone
2   Viettel           Khác       Viettel
3  Mobifone           Khác      Mobifone
4   Viettel           Khác       Viettel


---
## 3. `mart_revenue`
**Grain:** 1 row = 1 month  
Monthly GMV, revenue, user counts and growth rates.

In [ ]:
mart_revenue = (
    df.groupby('month', sort=True)
      .agg(
          total_txn    = ('order_id',  'count'),
          total_amount = ('Amount',    'sum'),
          revenue      = ('Revenue',   'sum'),
          active_users = ('user_id',   'nunique'),
          new_users    = ('user_id',   lambda x: x[df.loc[x.index, 'Type_user'] == 'New'].nunique()),
      )
      .reset_index()
)

# ── derived metrics ───────────────────────────────────────────────────────────
mart_revenue['returning_users']     = mart_revenue['active_users'] - mart_revenue['new_users']
mart_revenue['AOV']                 = mart_revenue['total_amount'] / mart_revenue['total_txn']
mart_revenue['revenue_per_user']    = mart_revenue['revenue']      / mart_revenue['active_users']
mart_revenue['take_rate']           = mart_revenue['revenue']      / mart_revenue['total_amount']
mart_revenue['pct_new_users']       = mart_revenue['new_users']    / mart_revenue['active_users']

# MoM growth (pct_change on sorted month)
mart_revenue['mom_gmv_growth']      = mart_revenue['total_amount'].pct_change()
mart_revenue['mom_revenue_growth']  = mart_revenue['revenue'].pct_change()

mart_revenue

,month,total_txn,total_amount,revenue,active_users,new_users,returning_users,AOV,revenue_per_user,take_rate,pct_new_users,mom_gmv_growth,mom_revenue_growth
0,2020-01,1000,53814234.0,1409827.02,1000,85,915,53814.234000,1409.827020,0.026198,0.085000,NaN,NaN
1,2020-02,972,52680000.0,1378500.00,969,91,878,54197.530864,1422.600619,0.026167,0.093911,-0.021077,-0.022220
2,2020-03,1098,58200000.0,1584000.00,1098,113,985,53005.464481,1442.622951,0.027216,0.102914,0.104784,0.149075
3,2020-04,1027,54150000.0,1488300.00,1027,81,946,52726.387537,1449.172347,0.027485,0.078870,-0.069588,-0.060417
4,2020-05,1104,52950000.0,1463700.00,1104,72,1032,47961.956522,1325.815217,0.027643,0.065217,-0.022161,-0.016529
5,2020-06,1118,58250000.0,1617200.00,1116,62,1054,52101.967800,1449.103943,0.027763,0.055556,0.100094,0.104871
6,2020-07,1153,58940000.0,1581900.00,1149,72,1077,51118.820468,1376.762402,0.026839,0.062663,0.011845,-0.021828
7,2020-08,1153,59950000.0,1618700.00,1153,84,1069,51994.796184,1403.902862,0.027001,0.072853,0.017136,0.023263
8,2020-09,1189,63400000.0,1702200.00,1189,82,1107,53322.119428,1431.623213,0.026849,0.068966,0.057548,0.051585
9,2020-10,1211,61690000.0,1690900.00,1209,79,1130,50941.370768,1398.593879,0.027410,0.065343,-0.026972,-0.006638


---
## 4. `mart_carrier`
**Grain:** 1 row = month × carrier  
Gmobile merged into 'Others'. Shares calculated against `mart_revenue` monthly totals.

In [10]:
# ── base aggregation by month × carrier_group ─────────────────────────────────
mart_carrier = (
    df.groupby(['month', 'carrier_group'], sort=True, observed=True)
      .agg(
          total_txn    = ('order_id', 'count'),
          total_amount = ('Amount',   'sum'),
          revenue      = ('Revenue',  'sum'),
          users        = ('user_id',  'nunique'),
      )
      .reset_index()
      .rename(columns={'carrier_group': 'carrier'})
)

# ── per-carrier derived metrics ───────────────────────────────────────────────
mart_carrier['AOV']       = mart_carrier['total_amount'] / mart_carrier['total_txn']
mart_carrier['take_rate'] = mart_carrier['revenue']      / mart_carrier['total_amount']

# ── merge monthly totals from mart_revenue to compute shares ─────────────────
monthly_totals = mart_revenue[['month', 'total_txn', 'total_amount']].rename(
    columns={'total_txn': 'month_total_txn', 'total_amount': 'month_total_amount'}
)
mart_carrier = mart_carrier.merge(monthly_totals, on='month', how='left')

mart_carrier['txn_share']     = mart_carrier['total_txn']    / mart_carrier['month_total_txn']
mart_carrier['revenue_share'] = mart_carrier['revenue']      / mart_carrier['month_total_amount']

# drop temp helper cols
mart_carrier.drop(columns=['month_total_txn', 'month_total_amount'], inplace=True)

mart_carrier.head(10)

,month,carrier,total_txn,total_amount,revenue,users,AOV,take_rate,txn_share,revenue_share
0,2020-01,Mobifone,282,15214234.0,456427.02,282,53951.184397,0.03,0.282000,0.008482
1,2020-01,Vietnamobile,42,1140000.0,45600.00,42,27142.857143,0.04,0.042000,0.000847
2,2020-01,Viettel,513,29530000.0,590600.00,513,57563.352827,0.02,0.513000,0.010975
3,2020-01,Vinaphone,163,7930000.0,317200.00,163,48650.306748,0.04,0.163000,0.005894
4,2020-02,Mobifone,280,13570000.0,407100.00,278,48464.285714,0.03,0.288066,0.007728
5,2020-02,Others,1,50000.0,2000.00,1,50000.000000,0.04,0.001029,0.000038
6,2020-02,Vietnamobile,30,760000.0,30400.00,30,25333.333333,0.04,0.030864,0.000577
7,2020-02,Viettel,480,29650000.0,593000.00,480,61770.833333,0.02,0.493827,0.011257
8,2020-02,Vinaphone,181,8650000.0,346000.00,181,47790.055249,0.04,0.186214,0.006568
9,2020-03,Mobifone,320,17120000.0,513600.00,320,53500.000000,0.03,0.291439,0.008825


---
## 5. `mart_behavior`
**Grain:** 1 row = month × Type_user × Purchase_status  
Answers: Does 'Mua hộ' have higher AOV? Do new users buy for others more?

In [12]:
mart_behavior = (
    df.groupby(['month', 'Type_user', 'purchase_label'], sort=True, observed=True)
      .agg(
          users   = ('user_id',  'nunique'),
          txn     = ('order_id', 'count'),
          amount  = ('Amount',   'sum'),
          revenue = ('Revenue',  'sum'),
      )
      .reset_index()
      .rename(columns={'purchase_label': 'Purchase_status'})
)

mart_behavior['AOV'] = mart_behavior['amount'] / mart_behavior['txn']

mart_behavior.head(12)

,month,Type_user,Purchase_status,users,txn,amount,revenue,AOV
0,2020-01,Current,Khác,797,797,36120000.0,925200.00,45319.949812
1,2020-01,Current,Mua hộ,118,118,14014234.0,386927.02,118764.694915
2,2020-01,New,Khác,79,79,2560000.0,67100.00,32405.063291
3,2020-01,New,Mua hộ,6,6,1120000.0,30600.00,186666.666667
4,2020-02,Current,Khác,758,760,34660000.0,891700.00,45605.263158
5,2020-02,Current,Mua hộ,120,121,14140000.0,391300.00,116859.504132
6,2020-02,New,Khác,84,84,3140000.0,78100.00,37380.952381
7,2020-02,New,Mua hộ,7,7,740000.0,17400.00,105714.285714
8,2020-03,Current,Khác,840,840,36050000.0,964800.00,42916.666667
9,2020-03,Current,Mua hộ,145,145,17660000.0,508100.00,121793.103448


---
## 6. `mart_demographic`
**Grain:** 1 row = 1 user  
User-level summary. Slice by Age / Gender / Location / Cohort inside Power BI via relationships — **no month dimension here** to avoid combinatorial explosion.

In [14]:
def safe_mode(s):
    """Return the first mode value (most frequent carrier per user)."""
    m = s.mode()
    return m.iloc[0] if not m.empty else None

mart_demographic = (
    df.groupby('user_id', sort=False)
      .agg(
          Age           = ('Age',           'first'),
          Gender        = ('Gender',         'first'),
          Location      = ('Location',       'first'),
          cohort_year   = ('cohort_year',    'first'),
          First_tran_date = ('First_tran_date', 'first'),
          total_txn     = ('order_id',       'count'),
          total_amount  = ('Amount',         'sum'),
          revenue       = ('Revenue',        'sum'),
          active_months = ('month',          'nunique'),
          carrier_mode  = ('carrier',        safe_mode),
      )
      .reset_index()
)

mart_demographic['AOV'] = mart_demographic['total_amount'] / mart_demographic['total_txn']

print(mart_demographic.shape)
mart_demographic.head()

(13390, 12)


,user_id,Age,Gender,Location,cohort_year,First_tran_date,total_txn,total_amount,revenue,active_months,carrier_mode,AOV
0,21269588,>37,FEMALE,HN,2018,2018-12-11,1,10000.0,300.0,1,Mobifone,10000.0
1,28097592,>37,FEMALE,HN,2019,2019-12-30,1,20000.0,600.0,1,Mobifone,20000.0
2,47435144,18_to_22,FEMALE,HN,2019,2019-11-11,1,10000.0,200.0,1,Viettel,10000.0
3,29080935,18_to_22,FEMALE,HN,2019,2019-10-24,1,10000.0,300.0,1,Mobifone,10000.0
4,14591075,18_to_22,FEMALE,Other Cities,2019,2019-12-28,1,10000.0,200.0,1,Viettel,10000.0


---
## 7. `mart_simulation`
**Grain:** 1 row = 1 carrier × 1 scenario  

What-if: if MoMo raises cashback from 1% → proposed rates, how much extra cost? What volume increase is needed to break even?

```
current_cashback_rate = 1%  (all carriers)
proposed_rates        = Viettel 2%, Mobifone 2.5%, Vinaphone/Vietnamobile/Gmobile 3%

commission_revenue    = Amount × Rate_pct/100   (from carrier to MoMo)
cashback_cost         = Amount × cashback_rate  (from MoMo to user)
net_revenue           = commission_revenue − cashback_cost

delta_cashback        = proposed_cashback − current_cashback
delta_revenue         = −delta_cashback
breakeven_volume_increase = delta_cashback / net_commission_rate
```

In [16]:
# ── carrier-level totals (original carrier, NOT carrier_group) ────────────────
carrier_totals = (
    df.groupby('carrier', observed=True)
      .agg(
          total_amount     = ('Amount',  'sum'),
          current_revenue  = ('Revenue', 'sum'),   # Amount × Rate_pct/100
      )
      .reset_index()
)

# ── commission rate per carrier (Rate_pct / 100) ──────────────────────────────
commission_rate = (
    df.groupby('carrier', observed=True)['Rate_pct']
      .first() / 100
).rename('commission_rate')
carrier_totals = carrier_totals.merge(commission_rate.reset_index(), on='carrier')

# ── cashback policy config ────────────────────────────────────────────────────
current_cashback_rate = 0.01

proposed_rates = {
    'Viettel':      0.02,
    'Mobifone':     0.025,
    'Vinaphone':    0.03,
    'Vietnamobile': 0.03,
    'Gmobile':      0.03,
}

carrier_totals['current_rate']   = current_cashback_rate
carrier_totals['proposed_rate']  = carrier_totals['carrier'].map(proposed_rates)

# ── simulation calculations ───────────────────────────────────────────────────
carrier_totals['current_cashback']  = carrier_totals['total_amount'] * carrier_totals['current_rate']
carrier_totals['proposed_cashback'] = carrier_totals['total_amount'] * carrier_totals['proposed_rate']

carrier_totals['delta_cashback']    = carrier_totals['proposed_cashback'] - carrier_totals['current_cashback']
carrier_totals['delta_revenue']     = -carrier_totals['delta_cashback']   # MoMo bears this cost

# Net commission rate = commission_rate - current_cashback_rate
# breakeven: how much extra volume needed to restore current_revenue
# new_volume × net_rate = current_revenue  →  extra_volume = delta_cashback / net_rate
carrier_totals['net_commission_rate']       = carrier_totals['commission_rate'] - current_cashback_rate
carrier_totals['breakeven_volume_increase'] = carrier_totals['delta_cashback'] / carrier_totals['net_commission_rate']

# ── final column order ────────────────────────────────────────────────────────
mart_simulation = carrier_totals[[
    'carrier', 'current_rate', 'proposed_rate',
    'total_amount', 'current_revenue',
    'current_cashback', 'proposed_cashback',
    'delta_cashback', 'delta_revenue',
    'net_commission_rate', 'breakeven_volume_increase'
]]

mart_simulation

,carrier,current_rate,proposed_rate,total_amount,current_revenue,current_cashback,proposed_cashback,delta_cashback,delta_revenue,net_commission_rate,breakeven_volume_increase
0,Gmobile,0.01,0.030,80000.0,3200.00,800.00,2400.00,1600.00,-1600.00,0.03,5.333333e+04
1,Mobifone,0.01,0.025,191904234.0,5757127.02,1919042.34,4797605.85,2878563.51,-2878563.51,0.02,1.439282e+08
2,Vietnamobile,0.01,0.030,21680000.0,867200.00,216800.00,650400.00,433600.00,-433600.00,0.03,1.445333e+07
3,Viettel,0.01,0.020,357620000.0,7152400.00,3576200.00,7152400.00,3576200.00,-3576200.00,0.01,3.576200e+08
4,Vinaphone,0.01,0.030,123320000.0,4932800.00,1233200.00,3699600.00,2466400.00,-2466400.00,0.03,8.221333e+07


## 8. Output Analytics Marts

In [ ]:
mart_revenue.to_csv(MART_DIR / 'mart_revenue.csv', index=False, encoding='utf-8-sig')
mart_carrier.to_csv(MART_DIR / 'mart_carrier.csv', index=False, encoding='utf-8-sig')
mart_behavior.to_csv(MART_DIR / 'mart_behavior.csv', index=False, encoding='utf-8-sig')
mart_demographic.to_csv(MART_DIR / 'mart_demographic.csv', index=False, encoding='utf-8-sig')
mart_simulation.to_csv(MART_DIR / 'mart_simulation.csv', index=False, encoding='utf-8-sig')

---
## 9. Summary

| File | Rows (expected) | Description |
|---|---|---|
| `mart_revenue.csv` | 12 | Monthly KPIs |
| `mart_carrier.csv` | 12 × ≤5 carriers | Carrier performance + share |
| `mart_behavior.csv` | 12 × 2 × 2 | Mua hộ vs Khác, New vs Current |
| `mart_demographic.csv` | ~1,300 users | User-grain, slice in PBI |
| `mart_simulation.csv` | 5 | Cashback what-if per carrier |

In [18]:
for name, frame in {
    'mart_revenue':     mart_revenue,
    'mart_carrier':     mart_carrier,
    'mart_behavior':    mart_behavior,
    'mart_demographic': mart_demographic,
    'mart_simulation':  mart_simulation,
}.items():
    print(f"{name:22s}  {frame.shape[0]:>5} rows × {frame.shape[1]:>2} cols")

mart_revenue               12 rows × 13 cols
mart_carrier               51 rows × 10 cols
mart_behavior              48 rows ×  8 cols
mart_demographic        13390 rows × 12 cols
mart_simulation             5 rows × 11 cols
